In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

In [2]:
# Dataset paths
train_dir = "dataset/train"
val_dir = "dataset/val"
test_dir = "dataset/test"

# Image settings
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Data generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

# Load datasets
train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_data = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

test_data = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

print("Dataset loaded successfully!")

Found 4097 images belonging to 2 classes.
Found 404 images belonging to 2 classes.
Found 399 images belonging to 2 classes.
Dataset loaded successfully!


In [3]:
# Load ResNet50 base model
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze base model layers
base_model.trainable = False

# Build final model
model = Sequential([
    base_model,
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

# Compile model
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Show model summary
model.summary()

94781440/94765736 [==============================] - 8s 0us/step
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 resnet50 (Functional)       (None, 7, 7, 2048)        23587712  
                                                                 
 flatten (Flatten)           (None, 100352)            0         
                                                                 
 dense (Dense)               (None, 128)               12845184  
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense_1 (Dense)             (None, 1)                 129       
                                                                 
Total params: 36,433,025
Trainable params: 12,845,313
Non-trainable params: 23,587,712
____________________________________

In [4]:
# Train the model
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

Epoch 1/5
129/129 [==============================] - 359s 3s/step - loss: 0.7506 - accuracy: 0.5138 - val_loss: 0.6898 - val_accuracy: 0.6040
Epoch 2/5
129/129 [==============================] - 330s 3s/step - loss: 0.6886 - accuracy: 0.5138 - val_loss: 0.6879 - val_accuracy: 0.4158
Epoch 3/5
129/129 [==============================] - 348s 3s/step - loss: 0.6859 - accuracy: 0.5372 - val_loss: 0.6920 - val_accuracy: 0.4158
Epoch 4/5
129/129 [==============================] - 509s 4s/step - loss: 0.6791 - accuracy: 0.5487 - val_loss: 0.6637 - val_accuracy: 0.7574
Epoch 5/5
129/129 [==============================] - 380s 3s/step - loss: 0.6699 - accuracy: 0.5685 - val_loss: 0.6525 - val_accuracy: 0.7698


In [13]:
# Save trained model
model.save("bone_fracture_resnet50.keras")

print("Model saved successfully!")

Model saved successfully!


In [14]:
# Evaluate model on test dataset
test_loss, test_accuracy = model.evaluate(test_data)

print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

13/13 [==============================] - 31s 2s/step - loss: 0.6281 - accuracy: 0.7193
Test Accuracy: 71.93%
Test Loss: 0.6281


In [9]:
from tensorflow.keras.preprocessing import image
from PIL import Image

# Class labels
class_names = ['Not Fractured', 'Fractured']

def predict_fracture(img_path):
    # Load image
    img = Image.open(img_path).convert("RGB")
    img = img.resize((224, 224))

    # Convert image to array
    img_array = image.img_to_array(img)
    img_array = img_array / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # Prediction
    prediction = model.predict(img_array)[0][0]

    # Result
    if prediction >= 0.4:
        result = class_names[1]
    else:
        result = class_names[0]

    confidence = float(prediction) * 100

    print(f"Prediction: {result}")
    print(f"Confidence: {confidence:.2f}%")

In [15]:
predict_fracture(r"C:\BoneFractureClassification\dataset\test\fractured\2-rotated3-rotated2-rotated2 - Copy.jpg")

Prediction: Fractured
Confidence: 42.91%


In [12]:
history2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
129/129 [==============================] - 409s 3s/step - loss: 0.6635 - accuracy: 0.5858 - val_loss: 0.6462 - val_accuracy: 0.7748
Epoch 2/10
129/129 [==============================] - 509s 4s/step - loss: 0.6541 - accuracy: 0.5953 - val_loss: 0.6429 - val_accuracy: 0.7599
Epoch 3/10
129/129 [==============================] - 542s 4s/step - loss: 0.6648 - accuracy: 0.5699 - val_loss: 0.6365 - val_accuracy: 0.6955
Epoch 4/10
129/129 [==============================] - 575s 4s/step - loss: 0.6402 - accuracy: 0.6197 - val_loss: 0.6124 - val_accuracy: 0.7624
Epoch 5/10
129/129 [==============================] - 467s 4s/step - loss: 0.6423 - accuracy: 0.6192 - val_loss: 0.5978 - val_accuracy: 0.7698
Epoch 6/10
129/129 [==============================] - 344s 3s/step - loss: 0.6340 - accuracy: 0.6200 - val_loss: 0.6151 - val_accuracy: 0.7896
Epoch 7/10
129/129 [==============================] - 339s 3s/step - loss: 0.6409 - accuracy: 0.6170 - val_loss: 0.6088 - val_accuracy: 0.7723

In [16]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Stop training if validation loss stops improving
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# Save best model automatically
checkpoint = ModelCheckpoint(
    "best_bone_fracture_model.keras",
    monitor='val_accuracy',
    save_best_only=True
)

print("Callbacks created successfully!")

Callbacks created successfully!


In [17]:
# Enable fine-tuning
base_model.trainable = True

# Freeze early layers, train last 50 layers
for layer in base_model.layers[:-50]:
    layer.trainable = False

# Recompile model with lower learning rate
model.compile(
    optimizer=Adam(learning_rate=0.00001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Fine-tuning enabled!")

Fine-tuning enabled!


In [18]:
history_finetune = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=[early_stop, checkpoint]
)

Epoch 1/10
129/129 [==============================] - 677s 5s/step - loss: 0.7414 - accuracy: 0.6300 - val_loss: 4.5789 - val_accuracy: 0.4158
Epoch 2/10
129/129 [==============================] - 599s 5s/step - loss: 0.5808 - accuracy: 0.6949 - val_loss: 1.1086 - val_accuracy: 0.4183
Epoch 3/10
129/129 [==============================] - 726s 6s/step - loss: 0.5319 - accuracy: 0.7337 - val_loss: 0.4043 - val_accuracy: 0.8614
Epoch 4/10
129/129 [==============================] - 691s 5s/step - loss: 0.4692 - accuracy: 0.7835 - val_loss: 0.2924 - val_accuracy: 0.8762
Epoch 5/10
129/129 [==============================] - 694s 5s/step - loss: 0.4215 - accuracy: 0.8089 - val_loss: 0.2572 - val_accuracy: 0.8936
Epoch 6/10
129/129 [==============================] - 644s 5s/step - loss: 0.3727 - accuracy: 0.8348 - val_loss: 0.4330 - val_accuracy: 0.7970
Epoch 7/10
129/129 [==============================] - 588s 5s/step - loss: 0.3384 - accuracy: 0.8528 - val_loss: 0.1667 - val_accuracy: 0.9332

In [21]:
best_model = tf.keras.models.load_model("model/best_bone_fracture_model.keras")

test_loss, test_accuracy = best_model.evaluate(test_data)

print(f"Best Model Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Best Model Test Loss: {test_loss:.4f}")

13/13 [==============================] - 23s 2s/step - loss: 0.2069 - accuracy: 0.9273
Best Model Test Accuracy: 92.73%
Best Model Test Loss: 0.2069


In [22]:
import tensorflow as tf

model = tf.keras.models.load_model(
    "model/best_bone_fracture_model.keras"
)

model.summary()

print("\n\nBase Model Layers:\n")

base_model = model.get_layer("resnet50")

for i, layer in enumerate(base_model.layers[-20:]):
    print(i, layer.name, layer.output_shape)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 resnet50 (Functional)       (None, 7, 7, 2048)        23587712  
                                                                 
 flatten (Flatten)           (None, 100352)            0         
                                                                 
 dense (Dense)               (None, 128)               12845184  
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense_1 (Dense)             (None, 1)                 129       
                                                                 
Total params: 36,433,025
Trainable params: 29,795,841
Non-trainable params: 6,637,184
_________________________________________________________________


Base Model Layers:

0 conv5_block2

In [23]:
import tensorflow as tf

model = tf.keras.models.load_model(
    "model/best_bone_fracture_model.keras"
)

base_model = model.get_layer("resnet50")

print("Input Shape:", model.input_shape)
print("Output Shape:", model.output_shape)

print("\nLast Conv Layer:")
print(base_model.get_layer("conv5_block3_out"))

Input Shape: (None, 224, 224, 3)
Output Shape: (None, 1)

Last Conv Layer:


In [24]:
import tensorflow as tf

model = tf.keras.models.load_model(
    "model/best_bone_fracture_model.keras"
)

base_model = model.get_layer("resnet50")

for i, layer in enumerate(base_model.layers[-30:]):
    print(i, layer.name, type(layer))

0 conv5_block1_1_relu <class 'keras.layers.core.activation.Activation'>
1 conv5_block1_2_conv <class 'keras.layers.convolutional.Conv2D'>
2 conv5_block1_2_bn <class 'keras.layers.normalization.batch_normalization.BatchNormalization'>
3 conv5_block1_2_relu <class 'keras.layers.core.activation.Activation'>
4 conv5_block1_0_conv <class 'keras.layers.convolutional.Conv2D'>
5 conv5_block1_3_conv <class 'keras.layers.convolutional.Conv2D'>
6 conv5_block1_0_bn <class 'keras.layers.normalization.batch_normalization.BatchNormalization'>
7 conv5_block1_3_bn <class 'keras.layers.normalization.batch_normalization.BatchNormalization'>
8 conv5_block1_add <class 'keras.layers.merge.Add'>
9 conv5_block1_out <class 'keras.layers.core.activation.Activation'>
10 conv5_block2_1_conv <class 'keras.layers.convolutional.Conv2D'>
11 conv5_block2_1_bn <class 'keras.layers.normalization.batch_normalization.BatchNormalization'>
12 conv5_block2_1_relu <class 'keras.layers.core.activation.Activation'>
13 conv5_blo